In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('car_fuel_efficiency_2026.csv')

df.head()

In [2]:
df = df[
    [
        'engine_displacement',
        'horsepower',
        'vehicle_weight',
        'model_year',
        'fuel_efficiency_mpg'
    ]
]

df.shape

(10000, 5)

In [3]:
df['fuel_efficiency_mpg'].describe()

count    10000.000000
mean        29.997700
std          2.930245
min         19.800000
25%         28.000000
50%         30.000000
75%         31.900000
max         41.200000
Name: fuel_efficiency_mpg, dtype: float64

In [ ]:
df['fuel_efficiency_mpg'].hist(bins=50)

In [5]:
df.isnull().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

In [6]:
df['horsepower'].median()

np.float64(254.0)

In [7]:
n = len(df)

n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)

idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

In [8]:
df_train.shape, df_val.shape, df_test.shape

((6000, 5), (2000, 5), (2000, 5))

In [9]:
features = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year'
]

X_train = df_train[features].values
X_val = df_val[features].values

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values

In [10]:
X_train_zero = np.nan_to_num(X_train, nan=0)
X_val_zero = np.nan_to_num(X_val, nan=0)

In [11]:
mean_hp = np.nanmean(X_train[:, 1])

In [12]:
X_train_mean = X_train.copy()
X_val_mean = X_val.copy()

X_train_mean[:, 1] = np.nan_to_num(
    X_train_mean[:, 1],
    nan=mean_hp
)

X_val_mean[:, 1] = np.nan_to_num(
    X_val_mean[:, 1],
    nan=mean_hp
)

In [14]:
mean_hp = np.nanmean(X_train[:, 1])
mean_hp

np.float64(254.46338337605272)

In [15]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T @ X
    XTX_inv = np.linalg.inv(XTX)

    w = XTX_inv @ X.T @ y

    return w

In [16]:
def predict_linear_regression(X, w):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    return X @ w

In [17]:
def rmse(y, y_pred):
    error = y_pred - y
    mse = np.mean(error ** 2)
    return np.sqrt(mse)

In [18]:
w_zero = train_linear_regression(X_train_zero, y_train)

In [19]:
y_pred_zero = predict_linear_regression(X_val_zero, w_zero)

In [20]:
rmse_zero = rmse(y_val, y_pred_zero)

round(rmse_zero, 3)

np.float64(2.205)

In [21]:
X_train_mean = X_train.copy()
X_val_mean = X_val.copy()

X_train_mean[:, 1] = np.nan_to_num(
    X_train_mean[:, 1],
    nan=mean_hp
)

X_val_mean[:, 1] = np.nan_to_num(
    X_val_mean[:, 1],
    nan=mean_hp
)

In [23]:
w_mean = train_linear_regression(X_train_mean, y_train)
y_pred_mean = predict_linear_regression(X_val_mean, w_mean)

In [24]:
rmse_mean = rmse(y_val, y_pred_mean)

round(rmse_mean, 3)

np.float64(2.202)

In [30]:
def train_linear_regression_reg(X, y, r=0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T @ X

    XTX = XTX + r * np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)

    w = XTX_inv @ X.T @ y

    return w

In [31]:
w = train_linear_regression_reg(X_train_zero, y_train, r=0)

print(w)
print(w.shape)

[-1.53884962e+02 -1.51505520e-03 -4.74024795e-06 -3.95418397e-03
  1.02146785e-01]
(5,)


In [32]:
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w = train_linear_regression_reg(X_train_zero, y_train, r=r)
    y_pred = predict_linear_regression(X_val_zero, w)
    score = rmse(y_val, y_pred)

    print(r, round(score, 4))

0 2.2053
0.01 2.2058
0.1 2.2241
1 2.3492
5 2.4094
10 2.4195
100 2.4292


In [33]:
scores = []

for seed in range(10):
    np.random.seed(seed)

    idx = np.arange(len(df))
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]

    X_train = df_train[features].values
    X_val = df_val[features].values

    y_train = df_train['fuel_efficiency_mpg'].values
    y_val = df_val['fuel_efficiency_mpg'].values

    X_train = np.nan_to_num(X_train, nan=0)
    X_val = np.nan_to_num(X_val, nan=0)

    w = train_linear_regression(X_train, y_train)

    y_pred = predict_linear_regression(X_val, w)

    score = rmse(y_val, y_pred)

    scores.append(score)

    print(seed, score)

0 2.2392041430547938
1 2.202112821083778
2 2.163419778753145
3 2.204358876855315
4 2.2018800943310333
5 2.245785150908199
6 2.2697212079518754
7 2.190794599936683
8 2.2166112016961836
9 2.207036592819637


In [34]:
np.std(scores)

np.float64(0.02878127407854454)

In [35]:
round(np.std(scores), 3)

np.float64(0.029)

In [36]:
np.random.seed(9)

idx = np.arange(len(df))
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

In [37]:
df_train_full = pd.concat([df_train, df_val])

In [38]:
df_train_full.shape, df_test.shape

((8000, 5), (2000, 5))

In [39]:
X_train_full = df_train_full[features].values
y_train_full = df_train_full['fuel_efficiency_mpg'].values

X_test = df_test[features].values
y_test = df_test['fuel_efficiency_mpg'].values

In [40]:
X_train_full = np.nan_to_num(X_train_full, nan=0)
X_test = np.nan_to_num(X_test, nan=0)

In [41]:
w_final = train_linear_regression_reg(
    X_train_full,
    y_train_full,
    r=0.001
)

In [42]:
y_pred_test = predict_linear_regression(
    X_test,
    w_final
)

In [43]:
test_rmse = rmse(y_test, y_pred_test)

test_rmse

np.float64(2.235827747196414)